# Coastal Flood 19 Audit: Around-Headland Links at Short Distance

This notebook checks whether mangrove-asset links can look spatially implausible (for example, around headlands) even when Euclidean distance is short.

It does **not** modify any existing notebooks or outputs.

## What this checks

1. Rebuild positive-avoided-EAD asset geometry exactly as in `coastal_flood_19`.
2. Link each positive asset to its nearest mangrove patch.
3. Compute diagnostics:
   - `nearest_mangrove_distance_m`
   - `% of shortest connector line over water`
   - `asset distance to coast`
   - `coastline arc distance / connector distance`
4. Flag potential around-headland links using transparent thresholds.
5. Summarize by short distance bins (0-250, 250-500, 500-1000, ...).
6. Optional focused check for the mangrove shown as `Label_No = 6` in the orange-line map reference table.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon

pd.options.display.max_columns = 200
pd.options.display.float_format = lambda x: f"{x:,.4f}"


In [ ]:
# Parameters
SCENARIO = 'minimum'  # 'minimum' or 'maximum'
FIXED_BUFFER_M = 5000.0

# Suspect-link diagnostic thresholds (for detecting around-headland behavior)
COASTAL_ASSET_MAX_DIST_M = 500.0
SUSPECT_MIN_WATER_SHARE = 0.50
SUSPECT_MIN_ARC_RATIO = 2.0

# Optional physically-oriented filter switch
STRICT_PHYSICAL_FILTER = True
STRICT_MAX_NEAREST_DISTANCE_M = 10000.0
STRICT_MAX_WATER_SHARE = 0.50
STRICT_MAX_ARC_RATIO = 2.0
STRICT_MAX_ANGLE_TO_NORMAL_DEG = 45.0

# Distance bins used in summaries
DIST_BINS = [0, 250, 500, 1000, 1500, 2000, 3000, 5000, np.inf]
DIST_LABELS = ['0-250', '250-500', '500-1000', '1000-1500', '1500-2000', '2000-3000', '3000-5000', '>5000']
SHORT_DISTANCE_LIMIT_M = 5000.0

FOCUS_LABEL_NO = 6  # from closest_mangroves_unattributed_reference_table_* csv

BASE = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
NETWORK_CSV = BASE / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
SHARED_INTERSECTIONS = BASE / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections'
MANGROVE_PATH = BASE / 'dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'
JAMAICA_BOUNDARY_PATH = BASE / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

RESULTS_PATH = BASE / f'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_{SCENARIO}_scenario'
ASSET_EAD_CSV = RESULTS_PATH / 'damage_estimates/coastal_ead_asset_level_usd.csv'
REFERENCE_LABEL_CSV = RESULTS_PATH / f'damage_estimates/mangrove_attribution_fixed_5000m/closest_mangroves_unattributed_reference_table_fixed_5000m_{SCENARIO}.csv'

OUT_DIR = RESULTS_PATH / 'damage_estimates/mangrove_attribution_fixed_5000m/headland_short_distance_audit'
OUT_DIR.mkdir(parents=True, exist_ok=True)

required = [NETWORK_CSV, SHARED_INTERSECTIONS, MANGROVE_PATH, JAMAICA_BOUNDARY_PATH, ASSET_EAD_CSV]
for q in required:
    if not q.exists():
        raise FileNotFoundError(q)

print('Scenario:', SCENARIO)
print('Output dir:', OUT_DIR)
print('Strict physical filter:', STRICT_PHYSICAL_FILTER)


In [ ]:
asset_key_cols = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']


def build_asset_gdf_for_scenario():
    network_details = pd.read_csv(NETWORK_CSV)
    network_details = network_details[['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']].drop_duplicates().copy()

    asset_ead = pd.read_csv(ASSET_EAD_CSV)
    map_layers = []
    missing_split_files = []

    for row in network_details.itertuples(index=False):
        split_file = SHARED_INTERSECTIONS / f"{row.asset_gpkg}_splits__coastal_flood_rasters_for_intersections__{row.asset_layer}.geoparquet"
        if not split_file.exists():
            missing_split_files.append(str(split_file))
            continue

        split_geom = gpd.read_parquet(split_file)
        if split_geom.crs is not None:
            split_geom = split_geom.to_crs('EPSG:3448')
        if row.asset_id_column not in split_geom.columns:
            continue

        split_geom = split_geom[[row.asset_id_column, 'geometry']].copy()
        split_geom = gpd.GeoDataFrame(split_geom, geometry='geometry', crs='EPSG:3448')

        ead_subset = asset_ead.loc[
            (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
            ['Asset_ID', 'Avoided_EAD_USD']
        ].copy()
        if ead_subset.empty:
            continue

        split_geom['_join_id'] = split_geom[row.asset_id_column].astype(str)
        ead_subset['_join_id'] = ead_subset['Asset_ID'].astype(str)

        merged = split_geom.merge(
            ead_subset[['_join_id', 'Avoided_EAD_USD']],
            on='_join_id',
            how='left'
        )

        merged['Sector'] = row.sector
        merged['Subsector'] = row.asset_description
        merged['Asset'] = row.asset_gpkg
        merged['Layer'] = row.asset_layer
        merged['Asset_ID'] = merged[row.asset_id_column].astype(str)
        merged['Avoided_EAD_USD'] = merged['Avoided_EAD_USD'].fillna(0.0)

        map_layers.append(merged[asset_key_cols + ['Avoided_EAD_USD', 'geometry']])

    if not map_layers:
        raise ValueError('No map layers built. Check inputs.')

    asset_gdf = gpd.GeoDataFrame(pd.concat(map_layers, ignore_index=True), geometry='geometry', crs='EPSG:3448')
    asset_gdf = asset_gdf.dissolve(by=asset_key_cols, as_index=False, aggfunc={'Avoided_EAD_USD': 'first'})
    asset_gdf = gpd.GeoDataFrame(asset_gdf, geometry='geometry', crs='EPSG:3448')

    return asset_gdf, sorted(set(missing_split_files))


def load_mangroves():
    mang = gpd.read_file(MANGROVE_PATH)
    if mang.crs is None:
        raise ValueError('Mangrove CRS missing.')
    if str(mang.crs).upper() != 'EPSG:3448':
        mang = mang.to_crs('EPSG:3448')

    if 'ID' in mang.columns:
        mang['Mangrove_ID'] = mang['ID'].astype(int)
    else:
        mang['Mangrove_ID'] = np.arange(1, len(mang) + 1)
    return mang


def load_main_jamaica_polygon():
    b = gpd.read_file(JAMAICA_BOUNDARY_PATH)
    if b.crs is None:
        raise ValueError('Jamaica boundary CRS missing.')
    if str(b.crs).upper() != 'EPSG:3448':
        b = b.to_crs('EPSG:3448')

    geom = b.union_all()
    if isinstance(geom, MultiPolygon):
        main_poly = max(geom.geoms, key=lambda p: p.area)
    elif isinstance(geom, Polygon):
        main_poly = geom
    else:
        polys = [g for g in getattr(geom, 'geoms', []) if isinstance(g, Polygon)]
        if not polys:
            raise ValueError('No polygon geometry in Jamaica boundary.')
        main_poly = max(polys, key=lambda p: p.area)
    return b, main_poly


In [ ]:
asset_gdf, missing_splits = build_asset_gdf_for_scenario()
mangroves = load_mangroves()
jamaica_boundary, jamaica_main_poly = load_main_jamaica_polygon()

positive_assets = asset_gdf.loc[asset_gdf['Avoided_EAD_USD'] > 0, asset_key_cols + ['Avoided_EAD_USD', 'geometry']].copy()
print('Positive-avoided assets:', len(positive_assets))
print('Mangrove patches:', len(mangroves))
print('Missing split files in build:', len(missing_splits))


In [ ]:
# Nearest mangrove link for each positive asset
nearest = gpd.sjoin_nearest(
    positive_assets,
    mangroves[['Mangrove_ID', 'geometry']],
    how='left',
    distance_col='nearest_mangrove_distance_m'
).drop(columns=['index_right'])

nearest = nearest.merge(
    mangroves[['Mangrove_ID', 'geometry']].rename(columns={'geometry': 'mangrove_geometry'}),
    on='Mangrove_ID',
    how='left'
)
nearest = gpd.GeoDataFrame(nearest, geometry='geometry', crs='EPSG:3448')

# Shortest connector line between asset geometry and mangrove polygon
connector = nearest.geometry.shortest_line(gpd.GeoSeries(nearest['mangrove_geometry'], crs='EPSG:3448'), align=False)

# Water/land split along connector using Jamaica main polygon
connector_len = connector.length
land_len = connector.intersection(jamaica_main_poly).length
water_len = (connector_len - land_len).clip(lower=0)
water_share = np.where(connector_len > 0, water_len / connector_len, 0.0)

# Coastal distance for asset points
coastline = jamaica_main_poly.boundary
asset_pts = nearest.geometry.representative_point()
mang_pts = gpd.GeoSeries(nearest['mangrove_geometry'], crs='EPSG:3448').representative_point()
asset_dist_to_coast = asset_pts.distance(coastline)

# Coastline arc distance (main shoreline) and ratio to direct connector
coast_ring = jamaica_main_poly.exterior
ring_len = coast_ring.length
arc_distance = []
for a, m in zip(asset_pts, mang_pts):
    sa = coast_ring.project(a)
    sm = coast_ring.project(m)
    d = abs(sa - sm)
    arc_distance.append(min(d, ring_len - d))
arc_distance = np.asarray(arc_distance, dtype=float)
arc_ratio = np.where(connector_len > 0, arc_distance / connector_len, np.nan)

# Angle between connector direction and local coast normal (degrees; 0 = perfectly coast-normal)
angle_to_normal = []
local_eps = min(200.0, max(5.0, 0.001 * ring_len))
for line, a in zip(connector, asset_pts):
    if line is None or line.is_empty:
        angle_to_normal.append(np.nan)
        continue
    coords = np.asarray(line.coords)
    if coords.shape[0] < 2:
        angle_to_normal.append(np.nan)
        continue

    # Direction from asset-side to mangrove-side along connector
    v = coords[-1] - coords[0]
    nv = np.linalg.norm(v)
    if nv == 0:
        angle_to_normal.append(np.nan)
        continue
    u = v / nv

    s = coast_ring.project(a)
    s0 = max(0.0, s - local_eps)
    s1 = min(ring_len, s + local_eps)
    c0 = np.asarray(coast_ring.interpolate(s0).coords[0])
    c1 = np.asarray(coast_ring.interpolate(s1).coords[0])

    t = c1 - c0
    nt = np.linalg.norm(t)
    if nt == 0:
        angle_to_normal.append(np.nan)
        continue
    t = t / nt

    n1 = np.array([-t[1], t[0]])
    n2 = -n1

    d1 = np.clip(np.dot(u, n1), -1.0, 1.0)
    d2 = np.clip(np.dot(u, n2), -1.0, 1.0)
    a1 = np.degrees(np.arccos(d1))
    a2 = np.degrees(np.arccos(d2))
    ang = min(a1, a2)
    if ang > 90.0:
        ang = 180.0 - ang
    angle_to_normal.append(ang)

angle_to_normal = np.asarray(angle_to_normal, dtype=float)

nearest['connector_len_m'] = connector_len
nearest['connector_water_len_m'] = water_len
nearest['connector_water_share'] = water_share
nearest['asset_dist_to_coast_m'] = asset_dist_to_coast
nearest['coast_arc_distance_m'] = arc_distance
nearest['arc_to_connector_ratio'] = arc_ratio
nearest['angle_to_coast_normal_deg'] = angle_to_normal

# Around-headland suspect diagnostic
nearest['suspect_headland'] = (
    (nearest['asset_dist_to_coast_m'] <= COASTAL_ASSET_MAX_DIST_M) &
    (nearest['nearest_mangrove_distance_m'] <= FIXED_BUFFER_M) &
    (nearest['connector_water_share'] >= SUSPECT_MIN_WATER_SHARE) &
    (nearest['arc_to_connector_ratio'] >= SUSPECT_MIN_ARC_RATIO)
)

# Optional physically-oriented keep filter
if STRICT_PHYSICAL_FILTER:
    nearest['strict_keep_physical'] = (
        (nearest['nearest_mangrove_distance_m'] <= STRICT_MAX_NEAREST_DISTANCE_M) &
        (nearest['connector_water_share'] <= STRICT_MAX_WATER_SHARE) &
        (nearest['arc_to_connector_ratio'] <= STRICT_MAX_ARC_RATIO) &
        (nearest['angle_to_coast_normal_deg'] <= STRICT_MAX_ANGLE_TO_NORMAL_DEG)
    )
else:
    nearest['strict_keep_physical'] = True

nearest['distance_bin'] = pd.cut(
    nearest['nearest_mangrove_distance_m'],
    bins=DIST_BINS,
    labels=DIST_LABELS,
    include_lowest=True,
    right=True,
)

nearest.head(5)


In [ ]:
# Summary by nearest-distance bins
rows = []
for label in DIST_LABELS:
    g = nearest.loc[nearest['distance_bin'] == label].copy()
    if g.empty:
        continue

    avoid_sum = float(g['Avoided_EAD_USD'].sum())
    suspect_avoid = float(g.loc[g['suspect_headland'], 'Avoided_EAD_USD'].sum())
    strict_keep_avoid = float(g.loc[g['strict_keep_physical'], 'Avoided_EAD_USD'].sum())

    rows.append({
        'distance_bin': label,
        'n_assets': int(len(g)),
        'avoid_usd_sum': avoid_sum,
        'suspect_n_assets': int(g['suspect_headland'].sum()),
        'suspect_pct_assets': float(100.0 * g['suspect_headland'].mean()),
        'suspect_avoid_usd': suspect_avoid,
        'suspect_pct_avoid': float(100.0 * suspect_avoid / avoid_sum) if avoid_sum != 0 else np.nan,
        'strict_keep_n_assets': int(g['strict_keep_physical'].sum()),
        'strict_keep_pct_assets': float(100.0 * g['strict_keep_physical'].mean()),
        'strict_keep_avoid_usd': strict_keep_avoid,
        'strict_keep_pct_avoid': float(100.0 * strict_keep_avoid / avoid_sum) if avoid_sum != 0 else np.nan,
        'mean_connector_water_share': float(g['connector_water_share'].mean()),
        'median_arc_to_connector_ratio': float(g['arc_to_connector_ratio'].median()),
        'median_angle_to_coast_normal_deg': float(g['angle_to_coast_normal_deg'].median()),
    })

summary_by_bin = pd.DataFrame(rows)
summary_by_bin


In [ ]:
overall = pd.DataFrame([{
    'scenario': SCENARIO,
    'n_positive_assets': int(len(nearest)),
    'n_suspect_headland': int(nearest['suspect_headland'].sum()),
    'suspect_pct_assets': float(100.0 * nearest['suspect_headland'].mean()),
    'total_avoid_usd': float(nearest['Avoided_EAD_USD'].sum()),
    'suspect_avoid_usd': float(nearest.loc[nearest['suspect_headland'], 'Avoided_EAD_USD'].sum()),
    'suspect_pct_avoid': float(100.0 * nearest.loc[nearest['suspect_headland'], 'Avoided_EAD_USD'].sum() / nearest['Avoided_EAD_USD'].sum()),
    'strict_keep_n_assets': int(nearest['strict_keep_physical'].sum()),
    'strict_keep_pct_assets': float(100.0 * nearest['strict_keep_physical'].mean()),
    'strict_keep_avoid_usd': float(nearest.loc[nearest['strict_keep_physical'], 'Avoided_EAD_USD'].sum()),
    'strict_keep_pct_avoid': float(100.0 * nearest.loc[nearest['strict_keep_physical'], 'Avoided_EAD_USD'].sum() / nearest['Avoided_EAD_USD'].sum()),
}])

print('Overall:')
display(overall)

short = nearest.loc[nearest['nearest_mangrove_distance_m'] <= SHORT_DISTANCE_LIMIT_M].copy()
short_summary = pd.DataFrame([{
    'scenario': SCENARIO,
    'short_distance_limit_m': SHORT_DISTANCE_LIMIT_M,
    'n_assets_short': int(len(short)),
    'avoid_usd_short': float(short['Avoided_EAD_USD'].sum()),
    'n_suspect_short': int(short['suspect_headland'].sum()),
    'suspect_pct_assets_short': float(100.0 * short['suspect_headland'].mean()) if len(short) > 0 else np.nan,
    'suspect_avoid_usd_short': float(short.loc[short['suspect_headland'], 'Avoided_EAD_USD'].sum()),
    'suspect_pct_avoid_short': float(100.0 * short.loc[short['suspect_headland'], 'Avoided_EAD_USD'].sum() / short['Avoided_EAD_USD'].sum()) if short['Avoided_EAD_USD'].sum() != 0 else np.nan,
    'strict_keep_n_short': int(short['strict_keep_physical'].sum()),
    'strict_keep_pct_assets_short': float(100.0 * short['strict_keep_physical'].mean()) if len(short) > 0 else np.nan,
    'strict_keep_avoid_short': float(short.loc[short['strict_keep_physical'], 'Avoided_EAD_USD'].sum()),
    'strict_keep_pct_avoid_short': float(100.0 * short.loc[short['strict_keep_physical'], 'Avoided_EAD_USD'].sum() / short['Avoided_EAD_USD'].sum()) if short['Avoided_EAD_USD'].sum() != 0 else np.nan,
}])

print('Short-distance check (<= 5 km by default):')
display(short_summary)

print('Top suspect links by avoided EAD:')
cols = [
    'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
    'Avoided_EAD_USD', 'Mangrove_ID', 'nearest_mangrove_distance_m',
    'asset_dist_to_coast_m', 'connector_water_share', 'arc_to_connector_ratio',
    'angle_to_coast_normal_deg', 'strict_keep_physical'
]
display(
    nearest.loc[nearest['suspect_headland'], cols]
    .sort_values('Avoided_EAD_USD', ascending=False)
    .head(25)
)


In [ ]:
# Focused check for Label_No = 6 in the orange-line reference table
focus_table = None
focus_mangrove_id = None
focus_assets = pd.DataFrame()

if REFERENCE_LABEL_CSV.exists():
    ref = pd.read_csv(REFERENCE_LABEL_CSV)
    focus_table = ref.loc[ref['Label_No'] == FOCUS_LABEL_NO].copy()
    if not focus_table.empty:
        focus_mangrove_id = int(focus_table['Mangrove_ID'].iloc[0])
        focus_assets = nearest.loc[nearest['Mangrove_ID'] == focus_mangrove_id].copy()

print('Reference table path exists:', REFERENCE_LABEL_CSV.exists())
if focus_table is not None and not focus_table.empty:
    print(f'Label {FOCUS_LABEL_NO} -> Mangrove_ID {focus_mangrove_id}')
    display(focus_table)

    if not focus_assets.empty:
        stats = pd.DataFrame([{
            'label_no': FOCUS_LABEL_NO,
            'mangrove_id': focus_mangrove_id,
            'n_linked_assets': int(len(focus_assets)),
            'avoid_usd_sum': float(focus_assets['Avoided_EAD_USD'].sum()),
            'mean_nearest_distance_m': float(focus_assets['nearest_mangrove_distance_m'].mean()),
            'n_within_500m': int((focus_assets['nearest_mangrove_distance_m'] <= 500).sum()),
            'n_within_1000m': int((focus_assets['nearest_mangrove_distance_m'] <= 1000).sum()),
            'n_suspect': int(focus_assets['suspect_headland'].sum()),
            'suspect_pct_assets': float(100.0 * focus_assets['suspect_headland'].mean()),
            'strict_keep_n': int(focus_assets['strict_keep_physical'].sum()),
            'strict_keep_pct_assets': float(100.0 * focus_assets['strict_keep_physical'].mean()),
        }])
        display(stats)

        display(
            focus_assets[[
                'Sector','Subsector','Asset','Layer','Asset_ID','Avoided_EAD_USD',
                'nearest_mangrove_distance_m','asset_dist_to_coast_m',
                'connector_water_share','arc_to_connector_ratio','angle_to_coast_normal_deg',
                'suspect_headland','strict_keep_physical'
            ]].sort_values('Avoided_EAD_USD', ascending=False).head(20)
        )
    else:
        print('No nearest-link assets found for this mangrove in positive avoided-EAD set.')
else:
    print('Label number not found in reference table for this scenario.')


In [ ]:
# Save outputs
pairs_out_cols = [
    'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD',
    'Mangrove_ID', 'nearest_mangrove_distance_m',
    'connector_len_m', 'connector_water_len_m', 'connector_water_share',
    'asset_dist_to_coast_m', 'coast_arc_distance_m', 'arc_to_connector_ratio',
    'angle_to_coast_normal_deg',
    'suspect_headland', 'strict_keep_physical', 'distance_bin', 'geometry'
]

pairs_gpkg = OUT_DIR / f'headland_audit_nearest_pairs_{SCENARIO}.gpkg'
pairs_csv = OUT_DIR / f'headland_audit_nearest_pairs_{SCENARIO}.csv'
summary_csv = OUT_DIR / f'headland_audit_summary_by_bin_{SCENARIO}.csv'
overall_csv = OUT_DIR / f'headland_audit_overall_{SCENARIO}.csv'
short_csv = OUT_DIR / f'headland_audit_short_distance_summary_{SCENARIO}.csv'
suspect_top_csv = OUT_DIR / f'headland_audit_top_suspects_{SCENARIO}.csv'
strict_dropped_short_csv = OUT_DIR / f'headland_audit_strict_dropped_short_distance_{SCENARIO}.csv'
focus_csv = OUT_DIR / f'headland_audit_focus_label_{FOCUS_LABEL_NO}_{SCENARIO}.csv'

nearest[pairs_out_cols].drop(columns=['geometry']).to_csv(pairs_csv, index=False)
summary_by_bin.to_csv(summary_csv, index=False)
overall.to_csv(overall_csv, index=False)
short_summary.to_csv(short_csv, index=False)

nearest.loc[nearest['suspect_headland']].sort_values('Avoided_EAD_USD', ascending=False).head(500)[
    ['Sector','Subsector','Asset','Layer','Asset_ID','Avoided_EAD_USD','Mangrove_ID','nearest_mangrove_distance_m','asset_dist_to_coast_m','connector_water_share','arc_to_connector_ratio','angle_to_coast_normal_deg','strict_keep_physical']
].to_csv(suspect_top_csv, index=False)

nearest.loc[(nearest['nearest_mangrove_distance_m'] <= SHORT_DISTANCE_LIMIT_M) & (~nearest['strict_keep_physical'])].sort_values(
    'Avoided_EAD_USD', ascending=False
)[[
    'Sector','Subsector','Asset','Layer','Asset_ID','Avoided_EAD_USD','Mangrove_ID','nearest_mangrove_distance_m',
    'asset_dist_to_coast_m','connector_water_share','arc_to_connector_ratio','angle_to_coast_normal_deg','suspect_headland'
]].to_csv(strict_dropped_short_csv, index=False)

if isinstance(focus_assets, pd.DataFrame) and not focus_assets.empty:
    focus_assets[[
        'Sector','Subsector','Asset','Layer','Asset_ID','Avoided_EAD_USD','Mangrove_ID',
        'nearest_mangrove_distance_m','asset_dist_to_coast_m','connector_water_share',
        'arc_to_connector_ratio','angle_to_coast_normal_deg','suspect_headland','strict_keep_physical'
    ]].to_csv(focus_csv, index=False)

# Save geometry output for GIS inspection
nearest[pairs_out_cols].to_file(pairs_gpkg, driver='GPKG')

print('Saved:')
print(' -', pairs_csv)
print(' -', pairs_gpkg)
print(' -', summary_csv)
print(' -', overall_csv)
print(' -', short_csv)
print(' -', suspect_top_csv)
print(' -', strict_dropped_short_csv)
if isinstance(focus_assets, pd.DataFrame) and not focus_assets.empty:
    print(' -', focus_csv)


In [ ]:
# Optional map: suspect links (nearest pair connectors)
plot = nearest.loc[nearest['suspect_headland']].copy()
if plot.empty:
    print('No suspect links under current thresholds.')
else:
    # Build line geometry for suspect links only
    suspect_lines = plot.geometry.shortest_line(gpd.GeoSeries(plot['mangrove_geometry'], crs='EPSG:3448'), align=False)
    suspect_gdf = gpd.GeoDataFrame(plot.copy(), geometry=suspect_lines, crs='EPSG:3448')

    fig, ax = plt.subplots(figsize=(12, 10))
    jamaica_boundary.boundary.plot(ax=ax, color='#999999', linewidth=0.4, zorder=1)
    mangroves.boundary.plot(ax=ax, color='#cccccc', linewidth=0.2, zorder=2)

    suspect_gdf.plot(ax=ax, color='#ff7f0e', linewidth=0.6, alpha=0.6, zorder=3)

    # Asset points sized by avoided EAD
    pts = gpd.GeoDataFrame(plot.copy(), geometry=plot.geometry.representative_point(), crs='EPSG:3448')
    size = 12 + 8 * np.log10(np.clip(pts['Avoided_EAD_USD'].values, 1e-6, None) + 1)
    pts.plot(ax=ax, color='#d500a3', markersize=size, alpha=0.85, zorder=4)

    ax.set_title(f'Suspect around-headland nearest links ({SCENARIO})')
    ax.set_axis_off()
    plt.tight_layout()

    out_png = OUT_DIR / f'headland_audit_suspect_links_map_{SCENARIO}.png'
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print('Saved:', out_png)
    plt.show()


In [ ]:
# Optional map: non-suspect avoided-EAD links and mangrove relation
# This uses already-computed nearest pairs, so it is lighter than rebuilding everything.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

SHOW_STRICT_KEEP_ONLY = False  # True: map only strict_keep_physical subset
SHORT_DISTANCE_ONLY = False    # True: map only links <= SHORT_DISTANCE_LIMIT_M
MAX_CONNECTORS_TO_PLOT = 8000  # sampling cap for line clutter/performance
RANDOM_SEED = 42

pairs_map = nearest.copy()
pairs_map = pairs_map.loc[pairs_map['Avoided_EAD_USD'] > 0].copy()
pairs_map = pairs_map.loc[~pairs_map['suspect_headland']].copy()

if SHOW_STRICT_KEEP_ONLY:
    pairs_map = pairs_map.loc[pairs_map['strict_keep_physical']].copy()

if SHORT_DISTANCE_ONLY:
    pairs_map = pairs_map.loc[pairs_map['nearest_mangrove_distance_m'] <= SHORT_DISTANCE_LIMIT_M].copy()

if pairs_map.empty:
    print('No rows in selected non-suspect subset with current switches.')
else:
    # Merge mangrove polygons for connector drawing
    mang_map = mangroves[['Mangrove_ID', 'geometry']].rename(columns={'geometry': 'mangrove_geometry'})
    if 'mangrove_geometry' in pairs_map.columns:
        pairs_map = pairs_map.drop(columns=['mangrove_geometry'])
    pairs_map = pairs_map.merge(mang_map, on='Mangrove_ID', how='left')
    pairs_map = pairs_map.dropna(subset=['mangrove_geometry']).copy()

    # Track full subset totals before sampling
    full_n = len(pairs_map)
    full_avoid = float(pairs_map['Avoided_EAD_USD'].sum())

    # Optional sampling for line plotting
    if len(pairs_map) > MAX_CONNECTORS_TO_PLOT:
        pairs_plot = pairs_map.sample(MAX_CONNECTORS_TO_PLOT, random_state=RANDOM_SEED).copy()
    else:
        pairs_plot = pairs_map.copy()

    # Build connectors (asset geometry -> nearest mangrove polygon)
    conn = pairs_plot.geometry.shortest_line(
        gpd.GeoSeries(pairs_plot['mangrove_geometry'], crs='EPSG:3448'),
        align=False,
    )
    conn_gdf = gpd.GeoDataFrame(pairs_plot.copy(), geometry=conn, crs='EPSG:3448')

    # Asset representative points
    pts = gpd.GeoDataFrame(pairs_plot.copy(), geometry=pairs_plot.geometry.representative_point(), crs='EPSG:3448')
    sizes = 10 + 8 * np.log10(np.clip(pts['Avoided_EAD_USD'].values, 1e-6, None) + 1)

    # Mangroves linked to selected subset
    linked_ids = sorted(set(pairs_plot['Mangrove_ID'].dropna().astype(int).tolist()))
    linked_mang = mangroves[mangroves['Mangrove_ID'].isin(linked_ids)].copy()

    fig, ax = plt.subplots(figsize=(14, 11))
    ax.set_facecolor('white')

    jamaica_boundary.boundary.plot(ax=ax, color='#9e9e9e', linewidth=0.4, zorder=1)
    mangroves.plot(ax=ax, facecolor='#e6e6e6', edgecolor='#b5b5b5', linewidth=0.15, alpha=0.65, zorder=2)
    linked_mang.plot(ax=ax, facecolor='#4f8dd8', edgecolor='#1f3e63', linewidth=0.35, alpha=0.9, zorder=3)

    conn_gdf.plot(ax=ax, color='#2aa6b8', linewidth=0.35, alpha=0.28, zorder=4)
    pts.plot(ax=ax, color='#8a2be2', markersize=sizes, alpha=0.72, zorder=5)

    title_bits = ['Non-suspect avoided-EAD links']
    if SHOW_STRICT_KEEP_ONLY:
        title_bits.append('strict-keep only')
    if SHORT_DISTANCE_ONLY:
        title_bits.append(f'<= {int(SHORT_DISTANCE_LIMIT_M)} m')
    title_txt = ' | '.join(title_bits) + f' ({SCENARIO})'

    ax.set_title(title_txt, fontsize=13)
    ax.set_axis_off()

    legend_handles = [
        Patch(facecolor='#e6e6e6', edgecolor='#b5b5b5', label='All mangroves (context)'),
        Patch(facecolor='#4f8dd8', edgecolor='#1f3e63', label='Mangroves linked to plotted assets'),
        Line2D([0], [0], color='#2aa6b8', lw=1.2, label='Asset -> nearest mangrove connector'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#8a2be2', markeredgecolor='#8a2be2', markersize=7, label='Non-suspect avoided-EAD assets')
    ]
    ax.legend(handles=legend_handles, loc='lower left', frameon=True, facecolor='white', framealpha=0.95)

    plt.tight_layout()

    mode_tag = 'strictkeep' if SHOW_STRICT_KEEP_ONLY else 'nonsuspect'
    dist_tag = 'short' if SHORT_DISTANCE_ONLY else 'all'
    out_png = OUT_DIR / f'headland_audit_{mode_tag}_links_map_{dist_tag}_{SCENARIO}.png'
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print('Saved:', out_png)
    print(f'Assets in full selected subset: {full_n:,}')
    print(f'Avoided EAD in full selected subset (USD): {full_avoid:,.2f}')
    print(f'Assets plotted (after sampling cap): {len(pairs_plot):,}')
    plt.show()


## Optional 4 km Odd-Link Check

This section checks whether some avoided-EAD links still look physically odd even when nearest mangrove distance is <= 4 km.


In [ ]:
# Optional check: physically odd links within <= 4 km
ODD_CHECK_MAX_DISTANCE_M = 4000.0
ODD_RELAXED_WATER_SHARE_MIN = 0.30
ODD_RELAXED_ARC_RATIO_MIN = 1.5
ODD_STRICT_OBLIQUE_MIN_DEG = 60.0

odd_4k = nearest.loc[(nearest['Avoided_EAD_USD'] > 0) & (nearest['nearest_mangrove_distance_m'] <= ODD_CHECK_MAX_DISTANCE_M)].copy()

odd_suspect_4k = odd_4k.loc[odd_4k['suspect_headland']].copy()
odd_borderline_non_sus_4k = odd_4k.loc[
    (~odd_4k['suspect_headland']) &
    (odd_4k['asset_dist_to_coast_m'] <= COASTAL_ASSET_MAX_DIST_M) &
    (odd_4k['connector_water_share'] >= ODD_RELAXED_WATER_SHARE_MIN) &
    (odd_4k['arc_to_connector_ratio'] >= ODD_RELAXED_ARC_RATIO_MIN)
].copy()

odd_very_strict_4k = odd_4k.loc[
    (odd_4k['asset_dist_to_coast_m'] <= COASTAL_ASSET_MAX_DIST_M) &
    (odd_4k['connector_water_share'] >= SUSPECT_MIN_WATER_SHARE) &
    (odd_4k['arc_to_connector_ratio'] >= SUSPECT_MIN_ARC_RATIO) &
    (odd_4k['angle_to_coast_normal_deg'] >= ODD_STRICT_OBLIQUE_MIN_DEG)
].copy()

avoid_total_4k = float(odd_4k['Avoided_EAD_USD'].sum())

odd_summary_4k = pd.DataFrame([{
    'scenario': SCENARIO,
    'distance_limit_m': ODD_CHECK_MAX_DISTANCE_M,
    'n_assets_le4km': int(len(odd_4k)),
    'avoid_usd_le4km': avoid_total_4k,
    'n_suspect_headland': int(len(odd_suspect_4k)),
    'pct_assets_suspect_headland': float(100.0 * len(odd_suspect_4k) / len(odd_4k)) if len(odd_4k) > 0 else np.nan,
    'avoid_usd_suspect_headland': float(odd_suspect_4k['Avoided_EAD_USD'].sum()),
    'pct_avoid_suspect_headland': float(100.0 * odd_suspect_4k['Avoided_EAD_USD'].sum() / avoid_total_4k) if avoid_total_4k != 0 else np.nan,
    'n_borderline_non_suspect': int(len(odd_borderline_non_sus_4k)),
    'pct_assets_borderline_non_suspect': float(100.0 * len(odd_borderline_non_sus_4k) / len(odd_4k)) if len(odd_4k) > 0 else np.nan,
    'avoid_usd_borderline_non_suspect': float(odd_borderline_non_sus_4k['Avoided_EAD_USD'].sum()),
    'pct_avoid_borderline_non_suspect': float(100.0 * odd_borderline_non_sus_4k['Avoided_EAD_USD'].sum() / avoid_total_4k) if avoid_total_4k != 0 else np.nan,
    'n_very_strict_odd': int(len(odd_very_strict_4k)),
    'pct_assets_very_strict_odd': float(100.0 * len(odd_very_strict_4k) / len(odd_4k)) if len(odd_4k) > 0 else np.nan,
    'avoid_usd_very_strict_odd': float(odd_very_strict_4k['Avoided_EAD_USD'].sum()),
    'pct_avoid_very_strict_odd': float(100.0 * odd_very_strict_4k['Avoided_EAD_USD'].sum() / avoid_total_4k) if avoid_total_4k != 0 else np.nan,
}])

odd_summary_csv = OUT_DIR / f'headland_audit_oddlinks_le4000m_summary_{SCENARIO}.csv'
odd_summary_4k.to_csv(odd_summary_csv, index=False)

border_cols = [
    'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD', 'Mangrove_ID',
    'nearest_mangrove_distance_m', 'asset_dist_to_coast_m', 'connector_water_share',
    'arc_to_connector_ratio', 'angle_to_coast_normal_deg'
]
odd_border_pts = gpd.GeoDataFrame(odd_borderline_non_sus_4k.copy(), geometry=odd_borderline_non_sus_4k.geometry.representative_point(), crs='EPSG:3448').to_crs('EPSG:4326')
odd_border_export = odd_border_pts[border_cols].copy()
odd_border_export['lat'] = odd_border_pts.geometry.y
odd_border_export['lon'] = odd_border_pts.geometry.x
odd_border_csv = OUT_DIR / f'headland_audit_oddlinks_le4000m_borderline_non_suspect_{SCENARIO}.csv'
odd_border_export.sort_values(['nearest_mangrove_distance_m', 'connector_water_share', 'arc_to_connector_ratio'], ascending=[False, False, False]).to_csv(odd_border_csv, index=False)

print('<=4 km odd-link summary:')
display(odd_summary_4k)
print('Saved:', odd_summary_csv)
print('Saved:', odd_border_csv)


In [ ]:
# Optional map: odd-link candidates at <= 4 km
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

odd_map = gpd.GeoDataFrame(pd.concat([
    odd_suspect_4k.assign(odd_flag='suspect_headland'),
    odd_borderline_non_sus_4k.assign(odd_flag='borderline_non_suspect')
], ignore_index=True), geometry='geometry', crs='EPSG:3448')

if odd_map.empty:
    print('No <=4 km odd-link candidates under current thresholds.')
else:
    odd_mang = mangroves[['Mangrove_ID', 'geometry']].rename(columns={'geometry': 'mangrove_geometry'})
    if 'mangrove_geometry' in odd_map.columns:
        odd_map = odd_map.drop(columns=['mangrove_geometry'])
    odd_map = odd_map.merge(odd_mang, on='Mangrove_ID', how='left')
    odd_map = odd_map.dropna(subset=['mangrove_geometry']).copy()

    odd_conn = odd_map.geometry.shortest_line(gpd.GeoSeries(odd_map['mangrove_geometry'], crs='EPSG:3448'), align=False)
    odd_conn_gdf = gpd.GeoDataFrame(odd_map[['odd_flag', 'Avoided_EAD_USD']].copy(), geometry=odd_conn, crs='EPSG:3448')

    odd_pts = gpd.GeoDataFrame(odd_map.copy(), geometry=odd_map.geometry.representative_point(), crs='EPSG:3448')
    odd_sizes = np.clip(6 + 4 * np.log10(np.clip(odd_pts['Avoided_EAD_USD'].values, 1e-9, None) + 1), 4, 16)

    linked_ids = sorted(set(odd_map['Mangrove_ID'].dropna().astype(int).tolist()))
    linked_mang = mangroves[mangroves['Mangrove_ID'].isin(linked_ids)].copy()

    fig, ax = plt.subplots(figsize=(12, 11))
    ax.set_facecolor('white')

    jamaica_boundary.boundary.plot(ax=ax, color='#9e9e9e', linewidth=0.4, zorder=1)
    mangroves.plot(ax=ax, facecolor='#efefef', edgecolor='#bbbbbb', linewidth=0.15, alpha=0.7, zorder=2)
    linked_mang.boundary.plot(ax=ax, color='#333333', linewidth=0.7, alpha=0.9, zorder=3)

    odd_conn_gdf.loc[odd_conn_gdf['odd_flag'] == 'suspect_headland'].plot(ax=ax, color='#d7301f', linewidth=0.45, alpha=0.35, zorder=4)
    odd_conn_gdf.loc[odd_conn_gdf['odd_flag'] == 'borderline_non_suspect'].plot(ax=ax, color='#fd8d3c', linewidth=0.4, alpha=0.28, zorder=4)

    mask = odd_pts['odd_flag'] == 'suspect_headland'
    odd_pts.loc[mask].plot(ax=ax, color='#d7301f', markersize=odd_sizes[mask.values], alpha=0.7, zorder=5)
    odd_pts.loc[~mask].plot(ax=ax, color='#fd8d3c', markersize=odd_sizes[(~mask).values], alpha=0.65, zorder=5)

    ax.set_title(f'<=4 km odd-link candidates (suspect + borderline non-suspect) ({SCENARIO})')
    ax.set_axis_off()

    legend_handles = [
        Patch(facecolor='#efefef', edgecolor='#bbbbbb', label='All mangroves (context)'),
        Line2D([0], [0], color='#333333', lw=1.2, label='Mangroves linked to odd candidates'),
        Line2D([0], [0], color='#d7301f', lw=1.2, label='Suspect headland link (strict)'),
        Line2D([0], [0], color='#fd8d3c', lw=1.2, label='Borderline non-suspect (relaxed criteria)'),
    ]
    ax.legend(handles=legend_handles, loc='lower left', frameon=True, framealpha=0.92)

    plt.tight_layout()
    odd_map_png = OUT_DIR / f'headland_audit_oddlinks_le4000m_map_{SCENARIO}.png'
    fig.savefig(odd_map_png, dpi=300, bbox_inches='tight')
    print('Saved:', odd_map_png)
    print('suspect_headland count:', len(odd_suspect_4k), 'avoid_usd:', float(odd_suspect_4k['Avoided_EAD_USD'].sum()))
    print('borderline_non_suspect count:', len(odd_borderline_non_sus_4k), 'avoid_usd:', float(odd_borderline_non_sus_4k['Avoided_EAD_USD'].sum()))
    plt.show()


## Optional Coast-Mangrove-Asset Angle Check

This section tests whether the direction from coastline to mangrove is similar to the direction from mangrove to asset (avoided-EAD link), including outlier label summaries.


In [ ]:
# Optional angle-consistency check (coast -> mangrove vs mangrove -> asset)
from shapely.geometry import Point

ANGLE_STRICT_NORMAL_MAX_DEG = 45.0
ANGLE_DIFF_PASS_1_DEG = 30.0
ANGLE_DIFF_PASS_2_DEG = 45.0
OUTLIER_MIN_DISTANCE_M = 5000.0

positive_angle = nearest.loc[nearest['Avoided_EAD_USD'] > 0].copy()

if positive_angle.empty:
    print('No positive avoided-EAD links available for angle check.')
else:
    coast_ring = jamaica_main_poly.exterior
    ring_len = coast_ring.length
    local_eps = min(200.0, max(5.0, 0.001 * ring_len))
    probe_m = 20.0

    asset_pts = positive_angle.geometry.representative_point()
    mang_pts = gpd.GeoSeries(positive_angle['mangrove_geometry'], crs='EPSG:3448').representative_point()

    def _ang(u, v):
        nu = np.linalg.norm(u)
        nv = np.linalg.norm(v)
        if nu <= 1e-8 or nv <= 1e-8:
            return np.nan
        d = np.clip(np.dot(u / nu, v / nv), -1.0, 1.0)
        return float(np.degrees(np.arccos(d)))

    angle_cm_vs_ma = []
    angle_inland_vs_cm = []
    angle_inland_vs_ma = []
    angle_diff_about_inland = []

    for mp, ap in zip(mang_pts, asset_pts):
        if mp is None or ap is None or mp.is_empty or ap.is_empty:
            angle_cm_vs_ma.append(np.nan)
            angle_inland_vs_cm.append(np.nan)
            angle_inland_vs_ma.append(np.nan)
            angle_diff_about_inland.append(np.nan)
            continue

        s = coast_ring.project(mp)
        cp = coast_ring.interpolate(s)

        v_cm = np.array([mp.x - cp.x, mp.y - cp.y], dtype=float)
        v_ma = np.array([ap.x - mp.x, ap.y - mp.y], dtype=float)

        s0 = max(0.0, s - local_eps)
        s1 = min(ring_len, s + local_eps)
        c0 = np.array(coast_ring.interpolate(s0).coords[0])
        c1 = np.array(coast_ring.interpolate(s1).coords[0])
        t = c1 - c0
        nt = np.linalg.norm(t)

        if nt <= 1e-8:
            a_cm_ma = _ang(v_cm, v_ma)
            angle_cm_vs_ma.append(a_cm_ma)
            angle_inland_vs_cm.append(np.nan)
            angle_inland_vs_ma.append(np.nan)
            angle_diff_about_inland.append(np.nan)
            continue

        t = t / nt
        n1 = np.array([-t[1], t[0]], dtype=float)
        n2 = -n1

        p1 = Point(cp.x + probe_m * n1[0], cp.y + probe_m * n1[1])
        p2 = Point(cp.x + probe_m * n2[0], cp.y + probe_m * n2[1])
        in1 = bool(jamaica_main_poly.contains(p1))
        in2 = bool(jamaica_main_poly.contains(p2))

        if in1 and not in2:
            ni = n1
        elif in2 and not in1:
            ni = n2
        else:
            # fallback if both/neither probe points are inside
            ni = n1 if np.dot(v_cm, n1) >= np.dot(v_cm, n2) else n2

        a_cm_ma = _ang(v_cm, v_ma)
        a_icm = _ang(ni, v_cm)
        a_ima = _ang(ni, v_ma)

        angle_cm_vs_ma.append(a_cm_ma)
        angle_inland_vs_cm.append(a_icm)
        angle_inland_vs_ma.append(a_ima)
        if np.isnan(a_icm) or np.isnan(a_ima):
            angle_diff_about_inland.append(np.nan)
        else:
            angle_diff_about_inland.append(abs(a_icm - a_ima))

    positive_angle['angle_cm_vs_ma_deg'] = np.asarray(angle_cm_vs_ma, dtype=float)
    positive_angle['angle_inlandnormal_vs_cm_deg'] = np.asarray(angle_inland_vs_cm, dtype=float)
    positive_angle['angle_inlandnormal_vs_ma_deg'] = np.asarray(angle_inland_vs_ma, dtype=float)
    positive_angle['angle_diff_cm_vs_ma_about_inlandnormal_deg'] = np.asarray(angle_diff_about_inland, dtype=float)

    positive_angle['pass_inland45_both'] = (
        (positive_angle['angle_inlandnormal_vs_cm_deg'] <= ANGLE_STRICT_NORMAL_MAX_DEG) &
        (positive_angle['angle_inlandnormal_vs_ma_deg'] <= ANGLE_STRICT_NORMAL_MAX_DEG)
    )
    positive_angle['pass_diff30'] = positive_angle['angle_diff_cm_vs_ma_about_inlandnormal_deg'] <= ANGLE_DIFF_PASS_1_DEG
    positive_angle['pass_diff45'] = positive_angle['angle_diff_cm_vs_ma_about_inlandnormal_deg'] <= ANGLE_DIFF_PASS_2_DEG

    # Push columns back to nearest for reuse downstream if needed
    angle_cols = [
        'angle_cm_vs_ma_deg',
        'angle_inlandnormal_vs_cm_deg',
        'angle_inlandnormal_vs_ma_deg',
        'angle_diff_cm_vs_ma_about_inlandnormal_deg',
        'pass_inland45_both', 'pass_diff30', 'pass_diff45'
    ]
    for c in angle_cols:
        nearest.loc[positive_angle.index, c] = positive_angle[c].values

    subsets = {
        'all_positive': positive_angle,
        '<=4km': positive_angle.loc[positive_angle['nearest_mangrove_distance_m'] <= 4000].copy(),
        '>5km': positive_angle.loc[positive_angle['nearest_mangrove_distance_m'] > OUTLIER_MIN_DISTANCE_M].copy(),
        'strict_keep': positive_angle.loc[positive_angle['strict_keep_physical']].copy(),
        'suspect_headland': positive_angle.loc[positive_angle['suspect_headland']].copy(),
    }

    subset_rows = []
    for name, g in subsets.items():
        if g.empty:
            continue
        subset_rows.append({
            'subset': name,
            'n_assets': int(len(g)),
            'avoid_sum_usd': float(g['Avoided_EAD_USD'].sum()),
            'median_angle_diff_deg': float(np.nanmedian(g['angle_diff_cm_vs_ma_about_inlandnormal_deg'])),
            'pct_pass_inland45_both': float(100.0 * np.nanmean(g['pass_inland45_both'])),
            'pct_pass_diff30': float(100.0 * np.nanmean(g['pass_diff30'])),
            'pct_pass_diff45': float(100.0 * np.nanmean(g['pass_diff45'])),
        })
    angle_subset_summary = pd.DataFrame(subset_rows)

    label_table = pd.read_csv(REFERENCE_LABEL_CSV)
    if {'Label_No', 'Mangrove_ID'}.issubset(label_table.columns):
        label_map = label_table[['Label_No', 'Mangrove_ID']].drop_duplicates().copy()
        label_map['Label_No'] = pd.to_numeric(label_map['Label_No'], errors='coerce')
        label_map['Mangrove_ID'] = pd.to_numeric(label_map['Mangrove_ID'], errors='coerce')
        label_map = label_map.dropna().astype({'Label_No': 'int64', 'Mangrove_ID': 'int64'})

        out = positive_angle.loc[positive_angle['nearest_mangrove_distance_m'] > OUTLIER_MIN_DISTANCE_M].copy()
        out = out.merge(label_map, on='Mangrove_ID', how='left')

        label_rows = []
        for label, g in out.dropna(subset=['Label_No']).groupby('Label_No'):
            label_rows.append({
                'Label_No': int(label),
                'n_assets': int(len(g)),
                'avoid_sum_usd': float(g['Avoided_EAD_USD'].sum()),
                'median_angle_diff_deg': float(np.nanmedian(g['angle_diff_cm_vs_ma_about_inlandnormal_deg'])),
                'pct_pass_inland45_both': float(100.0 * np.nanmean(g['pass_inland45_both'])),
                'pct_pass_diff30': float(100.0 * np.nanmean(g['pass_diff30'])),
                'pct_pass_diff45': float(100.0 * np.nanmean(g['pass_diff45'])),
            })
        angle_label_summary = pd.DataFrame(label_rows).sort_values('Label_No') if label_rows else pd.DataFrame()
    else:
        angle_label_summary = pd.DataFrame()

    out_subset_csv = OUT_DIR / f'headland_audit_angle_inlandnormal_subset_summary_{SCENARIO}.csv'
    out_label_csv = OUT_DIR / f'headland_audit_angle_inlandnormal_outlier_labels_{SCENARIO}.csv'
    out_pairs_csv = OUT_DIR / f'headland_audit_angle_inlandnormal_pairs_{SCENARIO}.csv'

    angle_subset_summary.to_csv(out_subset_csv, index=False)
    if not angle_label_summary.empty:
        angle_label_summary.to_csv(out_label_csv, index=False)

    positive_angle[[
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Mangrove_ID', 'Avoided_EAD_USD',
        'nearest_mangrove_distance_m', 'suspect_headland', 'strict_keep_physical',
        'angle_cm_vs_ma_deg', 'angle_inlandnormal_vs_cm_deg', 'angle_inlandnormal_vs_ma_deg',
        'angle_diff_cm_vs_ma_about_inlandnormal_deg', 'pass_inland45_both', 'pass_diff30', 'pass_diff45'
    ]].to_csv(out_pairs_csv, index=False)

    print('Angle subset summary:')
    display(angle_subset_summary)
    if not angle_label_summary.empty:
        print('Outlier label angle summary (>5km):')
        display(angle_label_summary)

    print('Saved:', out_subset_csv)
    if not angle_label_summary.empty:
        print('Saved:', out_label_csv)
    print('Saved:', out_pairs_csv)


## Optional Angle-Similarity Map

Map of coast-mangrove-asset angle consistency, with mangrove patch `Label_No` annotations from the outlier reference table.


In [ ]:
# Optional map: angle-similarity classes with patch numbering
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.patheffects as pe

ANGLE_MAP_POSITIVE_ONLY = True
ANGLE_MAP_OUTLIER_ONLY = True   # True: show only links with nearest distance > OUTLIER_MIN_DISTANCE_M
ANGLE_MAP_NON_SUSPECT_ONLY = False
ANGLE_MAP_MAX_CONNECTORS = 8000
ANGLE_SIMILAR_MAX_DEG = 30.0
ANGLE_MODERATE_MAX_DEG = 45.0

required_cols = ['angle_diff_cm_vs_ma_about_inlandnormal_deg']
missing_cols = [c for c in required_cols if c not in nearest.columns]
if missing_cols:
    raise ValueError(f'Missing angle columns: {missing_cols}. Run the angle-consistency cell first.')

angle_map_df = nearest.copy()
if ANGLE_MAP_POSITIVE_ONLY:
    angle_map_df = angle_map_df.loc[angle_map_df['Avoided_EAD_USD'] > 0].copy()
if ANGLE_MAP_OUTLIER_ONLY:
    angle_map_df = angle_map_df.loc[angle_map_df['nearest_mangrove_distance_m'] > OUTLIER_MIN_DISTANCE_M].copy()
if ANGLE_MAP_NON_SUSPECT_ONLY:
    angle_map_df = angle_map_df.loc[~angle_map_df['suspect_headland']].copy()

if angle_map_df.empty:
    print('No rows for angle-similarity map with current switches.')
else:
    # Attach label numbering for mangrove patches where available
    label_map = pd.read_csv(REFERENCE_LABEL_CSV)
    if {'Label_No', 'Mangrove_ID'}.issubset(label_map.columns):
        label_map = label_map[['Label_No', 'Mangrove_ID']].drop_duplicates().copy()
        label_map['Label_No'] = pd.to_numeric(label_map['Label_No'], errors='coerce')
        label_map['Mangrove_ID'] = pd.to_numeric(label_map['Mangrove_ID'], errors='coerce')
        label_map = label_map.dropna().astype({'Label_No': 'int64', 'Mangrove_ID': 'int64'})
    else:
        label_map = pd.DataFrame(columns=['Label_No', 'Mangrove_ID'])

    angle_map_df = angle_map_df.merge(label_map, on='Mangrove_ID', how='left')

    # Angle-similarity class
    a = angle_map_df['angle_diff_cm_vs_ma_about_inlandnormal_deg']
    angle_map_df['angle_class'] = np.select(
        [a <= ANGLE_SIMILAR_MAX_DEG, a <= ANGLE_MODERATE_MAX_DEG],
        ['similar', 'moderate'],
        default='off'
    )

    class_colors = {'similar': '#2ca25f', 'moderate': '#fec44f', 'off': '#de2d26'}

    # Merge mangrove geometry for connector drawing
    mang_map = mangroves[['Mangrove_ID', 'geometry']].rename(columns={'geometry': 'mangrove_geometry'})
    if 'mangrove_geometry' in angle_map_df.columns:
        angle_map_df = angle_map_df.drop(columns=['mangrove_geometry'])
    angle_map_df = angle_map_df.merge(mang_map, on='Mangrove_ID', how='left')
    angle_map_df = angle_map_df.dropna(subset=['mangrove_geometry']).copy()

    full_n = len(angle_map_df)
    full_avoid = float(angle_map_df['Avoided_EAD_USD'].sum())

    if len(angle_map_df) > ANGLE_MAP_MAX_CONNECTORS:
        angle_plot = angle_map_df.sample(ANGLE_MAP_MAX_CONNECTORS, random_state=42).copy()
    else:
        angle_plot = angle_map_df.copy()

    conn = angle_plot.geometry.shortest_line(
        gpd.GeoSeries(angle_plot['mangrove_geometry'], crs='EPSG:3448'),
        align=False
    )
    conn_gdf = gpd.GeoDataFrame(angle_plot[['angle_class', 'Avoided_EAD_USD']].copy(), geometry=conn, crs='EPSG:3448')

    pts = gpd.GeoDataFrame(angle_plot.copy(), geometry=angle_plot.geometry.representative_point(), crs='EPSG:3448')
    sizes = np.clip(6 + 4 * np.log10(np.clip(pts['Avoided_EAD_USD'].values, 1e-9, None) + 1), 4, 18)
    pts['plot_size'] = sizes

    # Labeled mangrove patches linked in current subset
    linked_ids = sorted(set(angle_map_df['Mangrove_ID'].dropna().astype(int).tolist()))
    linked_mang = mangroves[mangroves['Mangrove_ID'].isin(linked_ids)].copy()
    linked_labeled = linked_mang.merge(label_map, on='Mangrove_ID', how='left')

    fig, ax = plt.subplots(figsize=(14, 11))
    ax.set_facecolor('white')

    jamaica_boundary.boundary.plot(ax=ax, color='#9e9e9e', linewidth=0.4, zorder=1)
    mangroves.plot(ax=ax, facecolor='#efefef', edgecolor='#c0c0c0', linewidth=0.12, alpha=0.65, zorder=2)
    linked_mang.boundary.plot(ax=ax, color='#2f2f2f', linewidth=0.45, alpha=0.85, zorder=3)

    for cls in ['off', 'moderate', 'similar']:
        subc = conn_gdf.loc[conn_gdf['angle_class'] == cls]
        if not subc.empty:
            subc.plot(ax=ax, color=class_colors[cls], linewidth=0.45, alpha=0.35, zorder=4)

    for cls in ['off', 'moderate', 'similar']:
        subp = pts.loc[pts['angle_class'] == cls]
        if not subp.empty:
            subp.plot(ax=ax, color=class_colors[cls], markersize=subp['plot_size'], alpha=0.72, zorder=5)

    # Patch numbering labels
    ann = linked_labeled.dropna(subset=['Label_No']).copy()
    if not ann.empty:
        ann_pts = ann.geometry.representative_point()
        for lbl, pt in zip(ann['Label_No'].astype(int).values, ann_pts):
            txt = ax.text(pt.x, pt.y, str(lbl), fontsize=10, weight='bold', color='#111111', ha='center', va='center', zorder=6)
            txt.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])

    title_mode = '>5km only' if ANGLE_MAP_OUTLIER_ONLY else 'all distances'
    if ANGLE_MAP_NON_SUSPECT_ONLY:
        title_mode += ' | non-suspect only'
    ax.set_title(f'Coast-Mangrove-Asset angle similarity classes ({title_mode}, {SCENARIO})')
    ax.set_axis_off()

    legend_handles = [
        Patch(facecolor='#efefef', edgecolor='#c0c0c0', label='All mangroves (context)'),
        Line2D([0], [0], color='#2f2f2f', lw=1.2, label='Mangroves linked in current subset'),
        Line2D([0], [0], color=class_colors['similar'], lw=1.2, label=f'Similar (<= {ANGLE_SIMILAR_MAX_DEG:.0f} deg)'),
        Line2D([0], [0], color=class_colors['moderate'], lw=1.2, label=f'Moderate ({ANGLE_SIMILAR_MAX_DEG:.0f}-{ANGLE_MODERATE_MAX_DEG:.0f} deg)'),
        Line2D([0], [0], color=class_colors['off'], lw=1.2, label=f'Off (> {ANGLE_MODERATE_MAX_DEG:.0f} deg)'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#666666', markersize=6, label='Asset points (size ~ avoided EAD)'),
    ]
    ax.legend(handles=legend_handles, loc='lower left', frameon=True, facecolor='white', framealpha=0.95)

    plt.tight_layout()

    mode_tag = 'outliers' if ANGLE_MAP_OUTLIER_ONLY else 'all'
    ns_tag = 'nonsus' if ANGLE_MAP_NON_SUSPECT_ONLY else 'alllinks'
    map_png = OUT_DIR / f'headland_audit_angle_similarity_map_{mode_tag}_{ns_tag}_{SCENARIO}.png'
    map_csv = OUT_DIR / f'headland_audit_angle_similarity_classified_{mode_tag}_{ns_tag}_{SCENARIO}.csv'

    fig.savefig(map_png, dpi=300, bbox_inches='tight')

    angle_map_df[[
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Mangrove_ID', 'Label_No',
        'Avoided_EAD_USD', 'nearest_mangrove_distance_m',
        'angle_diff_cm_vs_ma_about_inlandnormal_deg', 'angle_class',
        'suspect_headland', 'strict_keep_physical'
    ]].to_csv(map_csv, index=False)

    print('Saved:', map_png)
    print('Saved:', map_csv)
    print(f'Assets in full subset: {full_n:,}')
    print(f'Avoided EAD in full subset (USD): {full_avoid:,.2f}')
    print('Class counts (full subset):')
    print(angle_map_df['angle_class'].value_counts(dropna=False))
    plt.show()


## Interpretation Notes

- This notebook is a diagnostic and does **not** modify existing analysis notebooks.
- It reports two things separately:
  1. `suspect_headland`: possible around-headland geometry issue
  2. `strict_keep_physical`: optional physically-oriented keep rule

### Strict physical keep rule (optional)
A link is kept when all are true:

1. nearest mangrove distance <= `STRICT_MAX_NEAREST_DISTANCE_M`
2. connector water share <= `STRICT_MAX_WATER_SHARE`
3. coastline-arc / direct ratio <= `STRICT_MAX_ARC_RATIO`
4. angle to coast-normal <= `STRICT_MAX_ANGLE_TO_NORMAL_DEG`

This rule is intentionally simple and transparent, so you can tune thresholds with your colleague.

### Short-distance concern
Use `short_summary` and `headland_audit_strict_dropped_short_distance_*.csv` to check whether short-distance links are also affected.
